Import PySpark functions

In [0]:
# ============================================================
# CELL 1 — IMPORT PYSPARK FUNCTIONS
# ============================================================

# Import PySpark SQL functions.
# We will use these functions to filter, select, and manipulate
# the pipeline metadata stored in Unity Catalog.
from pyspark.sql import functions as F

# Confirm that the notebook has loaded the required functions.
print("PySpark functions imported successfully.")

Read the metadata table

In [0]:
# ============================================================
# CELL 2 — READ PIPELINE METADATA
# ============================================================

# Read the pipeline metadata table from Unity Catalog.
#
# This table is our control table.
# It tells the framework what pipeline configuration
# should be processed.
metadata_df = spark.table(
    "workspace.nyc_taxi_audit.pipeline_metadata"
)

# Display the metadata so we can verify
# that the reader can access the control table.
display(metadata_df)

Select only active pipelines

In [0]:
# ============================================================
# CELL 3 — FILTER ACTIVE PIPELINES
# ============================================================

# Select only pipeline configurations where active_flag = true.
#
# This is important in a production metadata-driven framework:
# an inactive configuration should not be processed.
active_metadata_df = (
    metadata_df
    .filter(F.col("active_flag") == True)
)

# Display only the active pipeline configurations.
display(active_metadata_df)

Validate that active configuration exists

In [0]:
# ============================================================
# CELL 4 — VALIDATE ACTIVE PIPELINE CONFIGURATION
# ============================================================

# Count the number of active pipeline configurations.
active_pipeline_count = active_metadata_df.count()

# Check whether at least one active pipeline exists.
if active_pipeline_count == 0:

    # Stop the pipeline because there is nothing configured
    # for processing.
    raise ValueError(
        "No active pipeline configuration found."
    )

# Display a confirmation message when active configurations exist.
print(
    f"Active pipeline configurations found: "
    f"{active_pipeline_count}"
)

Select the configuration columns

In [0]:
# ============================================================
# CELL 5 — SELECT REQUIRED CONFIGURATION COLUMNS
# ============================================================

# Select only the columns that the ingestion framework
# will need to process a pipeline.
#
# Keeping this selection explicit makes the interface between
# the metadata reader and the ingestion engine clear.
pipeline_config_df = (
    active_metadata_df
    .select(
        "pipeline_id",
        "pipeline_name",
        "source_name",
        "source_path",
        "source_format",
        "source_file_pattern",
        "record_hash_columns",
        "incremental_strategy",
        "load_type",
        "target_catalog",
        "target_schema",
        "target_table",
        "dq_rule_set"
    )
)

# Display the final configuration that will be passed
# to the ingestion framework.
display(pipeline_config_df)

Convert configuration to a Python dictionary

In [0]:
# ============================================================
# CELL 6 — CONVERT PIPELINE CONFIGURATION TO DICTIONARY
# ============================================================

# Collect the first active pipeline configuration from Spark.
# We currently have one active pipeline, so taking the first
# row is appropriate for this development step.
pipeline_config_row = (
    pipeline_config_df
    .first()
)

# Make sure a configuration row was actually returned.
if pipeline_config_row is None:

    # Stop execution if no active configuration exists.
    raise ValueError(
        "No pipeline configuration available."
    )


# Convert the Spark Row into a standard Python dictionary.
# This makes the configuration easy to access inside
# reusable Python functions.
pipeline_config = pipeline_config_row.asDict()


# Display the configuration dictionary.
print("Pipeline configuration:")
print(pipeline_config)

Validate required configuration

In [0]:
# ============================================================
# CELL 7 — VALIDATE REQUIRED PIPELINE CONFIGURATION
# ============================================================

# Define the configuration fields that are mandatory
# for our metadata-driven ingestion framework.
required_fields = [
    "pipeline_id",
    "pipeline_name",
    "source_path",
    "source_format",
    "source_file_pattern",
    "record_hash_columns",
    "incremental_strategy",
    "load_type",
    "target_catalog",
    "target_schema",
    "target_table",
    "dq_rule_set"
]


# Create a list that will store any missing configuration fields.
missing_fields = []


# Check each required field one by one.
for field_name in required_fields:

    # Read the value from the pipeline configuration dictionary.
    field_value = pipeline_config.get(field_name)

    # Consider the field invalid when it is either:
    # 1. None
    # 2. An empty string
    if field_value is None or str(field_value).strip() == "":

        # Add the missing field name to our validation list.
        missing_fields.append(field_name)


# If any required fields are missing, fail immediately.
if missing_fields:

    # Create a useful error message containing all
    # missing configuration fields.
    raise ValueError(
        f"Missing required pipeline configuration fields: "
        f"{', '.join(missing_fields)}"
    )


# If execution reaches this point, the configuration
# passed all required-field checks.
print(
    f"Pipeline configuration validation successful: "
    f"{pipeline_config['pipeline_name']}"
)

Validate configuration values

In [0]:
# ============================================================
# CELL 8 — VALIDATE PIPELINE CONFIGURATION VALUES
# ============================================================

# Define the ingestion strategies that our framework supports.
# This prevents an invalid strategy from reaching the
# actual ingestion logic.
supported_incremental_strategies = [
    "FILE"
]


# Define the load types supported by our framework.
supported_load_types = [
    "FULL",
    "FILE_INCREMENTAL"
]


# Read the configured incremental strategy.
incremental_strategy = (
    pipeline_config["incremental_strategy"]
)


# Read the configured load type.
load_type = (
    pipeline_config["load_type"]
)


# Validate the incremental strategy.
if incremental_strategy not in supported_incremental_strategies:

    # Stop execution when an unsupported strategy is configured.
    raise ValueError(
        f"Unsupported incremental strategy: "
        f"{incremental_strategy}. "
        f"Supported values: "
        f"{supported_incremental_strategies}"
    )


# Validate the load type.
if load_type not in supported_load_types:

    # Stop execution when an unsupported load type is configured.
    raise ValueError(
        f"Unsupported load type: "
        f"{load_type}. "
        f"Supported values: "
        f"{supported_load_types}"
    )


# Validate the source format.
# We currently expect Parquet for our NYC Taxi pipeline.
if pipeline_config["source_format"].lower() != "parquet":

    # Stop execution if the configured source format
    # is not supported by this version of the framework.
    raise ValueError(
        "Unsupported source format. "
        "Expected: parquet"
    )


# Validate the target catalog.
if not pipeline_config["target_catalog"]:

    # Stop execution if a target catalog is not configured.
    raise ValueError(
        "Target catalog cannot be empty."
    )


# Validate the target schema.
if not pipeline_config["target_schema"]:

    # Stop execution if a target schema is not configured.
    raise ValueError(
        "Target schema cannot be empty."
    )


# Validate the target table.
if not pipeline_config["target_table"]:

    # Stop execution if a target table is not configured.
    raise ValueError(
        "Target table cannot be empty."
    )


# If every validation above passes, the configuration
# is valid for the current ingestion framework.
print(
    f"Pipeline configuration value validation successful: "
    f"{pipeline_config['pipeline_name']}"
)

Parse record_hash_columns

In [0]:
# ============================================================
# CELL 9 — PARSE RECORD HASH COLUMNS
# ============================================================

# Read the comma-separated hash-column configuration
# from the pipeline metadata.
record_hash_columns_string = (
    pipeline_config["record_hash_columns"]
)


# Convert the comma-separated string into a Python list.
#
# Example:
# "VendorID,tpep_pickup_datetime,trip_distance"
#
# becomes:
# ["VendorID", "tpep_pickup_datetime", "trip_distance"]
record_hash_columns = [
    column_name.strip()
    for column_name in record_hash_columns_string.split(",")
    if column_name.strip()
]


# Make sure at least one column has been configured
# for deterministic record ID generation.
if not record_hash_columns:

    # Stop execution when no hash columns are configured.
    raise ValueError(
        "No record hash columns configured "
        "for the pipeline."
    )


# Display the parsed list.
print("Record hash columns:")
print(record_hash_columns)

Validate the configured hash columns against the source

In [0]:
# ============================================================
# CELL 10 — VALIDATE RECORD HASH COLUMNS
# ============================================================

# Read the source path from the pipeline configuration.
# The path comes from metadata, so we are not hardcoding it.
source_path = pipeline_config["source_path"]


# Read the source data using the configured source format.
# The generic framework can later support other formats,
# but for our current project the source is Parquet.
if pipeline_config["source_format"].lower() == "parquet":

    # Read the Parquet source using the metadata-driven path.
    source_df = spark.read.parquet(source_path)

else:

    # Stop the pipeline when an unsupported source format
    # has been configured.
    raise ValueError(
        f"Unsupported source format: "
        f"{pipeline_config['source_format']}"
    )


# Get the actual column names available in the source data.
source_columns = set(source_df.columns)


# Find any configured hash columns that do not exist
# in the actual source schema.
missing_hash_columns = [
    column_name
    for column_name in record_hash_columns
    if column_name not in source_columns
]


# If any configured hash columns are missing,
# stop the pipeline before processing continues.
if missing_hash_columns:

    raise ValueError(
        "Configured record hash columns do not exist "
        f"in the source dataset: {missing_hash_columns}"
    )


# If execution reaches this point, every configured
# hash column exists in the source.
print(
    "Record hash column validation successful."
)


# Display the validated hash columns.
print(
    "Validated hash columns:",
    record_hash_columns
)

Validate source path

In [0]:
# ============================================================
# CELL 11 — VALIDATE SOURCE PATH
# ============================================================

# Read the source path from the pipeline configuration.
# The path is coming from metadata rather than being hardcoded.
source_path = pipeline_config["source_path"]


# Try to access the configured source location.
# dbutils.fs.ls() lists the files available at that path.
try:

    source_files = dbutils.fs.ls(source_path)

except Exception as error:

    # Stop execution when the configured source path
    # cannot be accessed.
    raise ValueError(
        f"Configured source path is not accessible: "
        f"{source_path}"
    ) from error


# Confirm that the configured source path is accessible.
print(
    f"Source path validation successful: {source_path}"
)


# Display the files available in the source location.
display(source_files)

Validate target table configuration

In [0]:
# ============================================================
# CELL 12 — VALIDATE TARGET TABLE CONFIGURATION
# ============================================================

# Read the target catalog, schema, and table name
# from the metadata-driven pipeline configuration.
target_catalog = pipeline_config["target_catalog"]
target_schema = pipeline_config["target_schema"]
target_table = pipeline_config["target_table"]


# Make sure the target catalog is not empty.
if not target_catalog:

    # Stop execution when the target catalog is missing.
    raise ValueError(
        "Target catalog is not configured."
    )


# Make sure the target schema is not empty.
if not target_schema:

    # Stop execution when the target schema is missing.
    raise ValueError(
        "Target schema is not configured."
    )


# Make sure the target table is not empty.
if not target_table:

    # Stop execution when the target table is missing.
    raise ValueError(
        "Target table is not configured."
    )


# Build the fully qualified Unity Catalog table name.
# Example:
# workspace.nyc_taxi_bronze.yellow_taxi
target_table_name = (
    f"{target_catalog}."
    f"{target_schema}."
    f"{target_table}"
)


# Display the final target table name.
print(
    f"Target table configuration validated: "
    f"{target_table_name}"
)

Validate source file pattern

In [0]:
# ============================================================
# CELL 13 — VALIDATE SOURCE FILE PATTERN
# ============================================================

# Read the file pattern from the pipeline metadata.
# The pattern tells the framework which files belong to
# this particular pipeline.
source_file_pattern = pipeline_config["source_file_pattern"]


# Make sure a file pattern has been configured.
if not source_file_pattern:

    # Stop the pipeline if the file pattern is missing.
    raise ValueError(
        "Source file pattern is not configured."
    )


# Get the files available in the configured source location.
# source_files was created in Cell 11.
available_file_names = [
    file_info.name
    for file_info in source_files
]


# Import Python's fnmatch module.
# It allows us to compare filenames against wildcard patterns
# such as: yellow_tripdata_*.parquet
import fnmatch


# Find files that match the configured pattern.
matching_files = [
    file_name
    for file_name in available_file_names
    if fnmatch.fnmatch(
        file_name,
        source_file_pattern
    )
]


# Stop the pipeline if the configured pattern does not
# match any available source file.
if not matching_files:

    raise ValueError(
        f"No source files found matching pattern: "
        f"{source_file_pattern}"
    )


# Display the successful validation result.
print(
    f"Source file pattern validation successful: "
    f"{source_file_pattern}"
)


# Display all matching files.
print("Matching source files:")
for file_name in matching_files:
    print(f" - {file_name}")